# Notebook 01 — Data Loading & Preprocessing
**Project:** Product Review Sentiment Classification  
**Student:** Eeman Khalid (22F-3173)  
**Datasets:**
- Primary: Amazon Product Reviews
- Secondary: Women's E-Commerce Clothing Reviews

This notebook cleans and saves the preprocessed datasets for model training.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install nltk --quiet

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter

for pkg in ['stopwords', 'wordnet', 'omw-1.4', 'punkt']:
    nltk.download(pkg, quiet=True)

STOP_WORDS = set(stopwords.words('english'))
NEGATIONS = {'no','not','nor','never','neither','barely','hardly','scarcely'}
STOP_WORDS -= NEGATIONS
lemmatizer = WordNetLemmatizer()

print('Libraries loaded.')

In [ ]:
# ── Kaggle dataset paths (update if needed) ────────────────────────────────
# After adding datasets in Kaggle, they appear under /kaggle/input/

AMAZON_PATH   = '/kaggle/input/amazon-product-reviews-dataset/1429_1.csv'
CLOTHING_PATH = '/kaggle/input/womens-ecommerce-clothing-reviews/Womens Clothing E-Commerce Reviews.csv'

OUTPUT_DIR = '/kaggle/working/data/processed/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ── Load Amazon dataset ────────────────────────────────────────────────────
amazon_raw = pd.read_csv(AMAZON_PATH)
print('Amazon shape:', amazon_raw.shape)
print('Columns:', list(amazon_raw.columns))
amazon_raw.head(3)

In [ ]:
# ── Identify text & rating columns ────────────────────────────────────────
text_col   = next(c for c in amazon_raw.columns if 'text' in c.lower())
rating_col = next(c for c in amazon_raw.columns if 'rating' in c.lower())
print(f'Text col: {text_col} | Rating col: {rating_col}')

amazon = amazon_raw[[text_col, rating_col]].copy()
amazon.columns = ['text', 'rating']
amazon.dropna(inplace=True)
amazon['rating'] = pd.to_numeric(amazon['rating'], errors='coerce')
amazon.dropna(subset=['rating'], inplace=True)
amazon['rating'] = amazon['rating'].astype(int)
amazon = amazon[amazon['rating'].between(1,5)]
print('After cleaning:', amazon.shape)

In [ ]:
# ── Label mapping ──────────────────────────────────────────────────────────
def rating_to_sentiment(r):
    if r == 5:       return 'positive'
    elif r in (3,4): return 'neutral'
    else:            return 'negative'

LABEL_MAP = {'positive': 2, 'neutral': 1, 'negative': 0}

amazon['sentiment'] = amazon['rating'].apply(rating_to_sentiment)
amazon['label']     = amazon['sentiment'].map(LABEL_MAP)

print('Class distribution:')
print(amazon['sentiment'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
amazon['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
amazon['sentiment'].value_counts().plot(kind='bar', ax=axes[1], color=['#28a745','#ffc107','#dc3545'])
axes[1].set_title('Sentiment Distribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'amazon_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── Text cleaning function ─────────────────────────────────────────────────
def clean_text(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS]
    return ' '.join(tokens)

# Test it
sample = "This product is NOT good at all!!! Terrible quality 😤"
print('Original:', sample)
print('Cleaned: ', clean_text(sample))

In [ ]:
# ── Apply cleaning ─────────────────────────────────────────────────────────
print('Cleaning Amazon reviews...')
amazon['clean_text'] = amazon['text'].apply(clean_text)

before = len(amazon)
amazon = amazon[amazon['clean_text'].str.strip() != '']
print(f'Dropped {before - len(amazon)} empty rows after cleaning.')
print('Final shape:', amazon.shape)
amazon.head(3)

In [ ]:
# ── Review length analysis ─────────────────────────────────────────────────
amazon['word_count'] = amazon['clean_text'].apply(lambda x: len(x.split()))

print(amazon['word_count'].describe())

plt.figure(figsize=(10, 4))
plt.hist(amazon['word_count'].clip(upper=200), bins=50, color='steelblue', edgecolor='white')
plt.title('Distribution of Review Lengths (word count, clipped at 200)')
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.axvline(amazon['word_count'].median(), color='red', linestyle='--', label=f"Median: {amazon['word_count'].median():.0f}")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'amazon_length_dist.png', dpi=150)
plt.show()

In [ ]:
# ── Save Amazon processed data ─────────────────────────────────────────────
amazon_save = amazon[['text', 'clean_text', 'rating', 'sentiment', 'label']]
amazon_save.to_csv(OUTPUT_DIR + 'amazon_clean.csv', index=False)
print(f'Saved amazon_clean.csv ({len(amazon_save):,} rows)')

In [ ]:
# ── Load & process Clothing dataset (secondary) ────────────────────────────
clothing_raw = pd.read_csv(CLOTHING_PATH)
print('Clothing shape:', clothing_raw.shape)
print('Columns:', list(clothing_raw.columns))

text_col_c   = next(c for c in clothing_raw.columns if 'review' in c.lower() and 'text' in c.lower())
rating_col_c = next(c for c in clothing_raw.columns if 'rating' in c.lower())

clothing = clothing_raw[[text_col_c, rating_col_c]].copy()
clothing.columns = ['text', 'rating']
clothing.dropna(inplace=True)
clothing['rating'] = pd.to_numeric(clothing['rating'], errors='coerce')
clothing.dropna(subset=['rating'], inplace=True)
clothing['rating'] = clothing['rating'].astype(int)
clothing = clothing[clothing['rating'].between(1,5)]

clothing['sentiment'] = clothing['rating'].apply(rating_to_sentiment)
clothing['label']     = clothing['sentiment'].map(LABEL_MAP)

print('\nClothing class distribution:')
print(clothing['sentiment'].value_counts())

print('\nCleaning clothing reviews...')
clothing['clean_text'] = clothing['text'].apply(clean_text)
clothing = clothing[clothing['clean_text'].str.strip() != '']

clothing[['text','clean_text','rating','sentiment','label']].to_csv(OUTPUT_DIR + 'clothing_clean.csv', index=False)
print(f'Saved clothing_clean.csv ({len(clothing):,} rows)')

In [ ]:
# ── Summary ────────────────────────────────────────────────────────────────
print('=' * 50)
print('PREPROCESSING COMPLETE')
print('=' * 50)
print(f'Amazon processed:   {len(amazon):,} rows')
print(f'Clothing processed: {len(clothing):,} rows')
print(f'\nOutput files saved to: {OUTPUT_DIR}')
print('\nNext: Run notebooks 02, 03, 04 for model training.')